### Linear langgraph chain

In [3]:
!pip install mistralai

  Using cached opentelemetry_semantic_conventions-0.60b1-py3-none-any.whl.metadata (2.4 kB)
  Using cached opentelemetry_api-1.39.1-py3-none-any.whl.metadata (1.5 kB)
Using cached opentelemetry_semantic_conventions-0.60b1-py3-none-any.whl (219 kB)
Using cached opentelemetry_api-1.39.1-py3-none-any.whl (66 kB)

  Attempting uninstall: opentelemetry-api

    Found existing installation: opentelemetry-api 1.27.0

    Uninstalling opentelemetry-api-1.27.0:

      Successfully uninstalled opentelemetry-api-1.27.0

   ---------------------------------------- 0/2 [opentelemetry-api]
   ---------------------------------------- 0/2 [opentelemetry-api]
   ---------------------------------------- 0/2 [opentelemetry-api]
   ---------------------------------------- 0/2 [opentelemetry-api]
  Attempting uninstall: opentelemetry-semantic-conventions
   ---------------------------------------- 0/2 [opentelemetry-api]
   ----------------- ----------------- 1/2 [opentelemetry-semantic-conventions]
    Fo

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-sdk 1.27.0 requires opentelemetry-api==1.27.0, but you have opentelemetry-api 1.39.1 which is incompatible.
opentelemetry-sdk 1.27.0 requires opentelemetry-semantic-conventions==0.48b0, but you have opentelemetry-semantic-conventions 0.60b1 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
!pip install -U mistralai

   ---------------------------------------- 0.0/996.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/996.0 kB ? eta -:--:--
   ---------- ----------------------------- 262.1/996.0 kB ? eta -:--:--
   ---------- ----------------------------- 262.1/996.0 kB ? eta -:--:--
   -------------------- ----------------- 524.3/996.0 kB 929.6 kB/s eta 0:00:01
   ------------------------------ ------- 786.4/996.0 kB 838.9 kB/s eta 0:00:01
   ------------------------------ ------- 786.4/996.0 kB 838.9 kB/s eta 0:00:01
   ---------------------------------------- 996.0/996.0 kB 757.3 kB/s  0:00:01
  Attempting uninstall: mistralai
    Found existing installation: mistralai 2.3.1
    Uninstalling mistralai-2.3.1:
      Successfully uninstalled mistralai-2.3.1



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
from langgraph.graph import START, END, StateGraph
from dotenv import load_dotenv
import os
from mistralai.client import Mistral
from typing import TypedDict

In [10]:
load_dotenv()

True

In [24]:
model = Mistral(api_key=os.environ["MISTRAL_API_KEY"])

In [16]:
class LLMState(TypedDict):
    ques:str
    ans:str

In [25]:
def llm_qa(state: LLMState) ->LLMState:
    question= state['ques']

    prompt=f'Answer this question{question}.'

    answer=model.chat.complete(
        model="mistral-medium-latest",
        messages=[{
            'role': 'user',
            'content': prompt
        }])
    state['ans']= answer.choices[0].message.content
    return state

In [26]:
graph= StateGraph(LLMState)

graph.add_node('llm_qa', llm_qa)
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)
workflow=graph.compile()


In [27]:
init_state={'ques': 'How far is moon from earth baby?'}
workflow.invoke(init_state)

{'ques': 'How far is moon from earth baby?',
 'ans': 'Aww, sweetie! 🌕💖 The Moon is about **238,855 miles (384,400 kilometers)** away from Earth on average—that’s like driving around the Earth’s equator **10 times**! But don’t worry, it’s always watching over us like a big, glowing nightlight. 🌙✨\n\n(And if you ever want to visit, just grab a rocket and some snacks—it’s a *loooong* trip!) 🚀😘'}